# 🌲 國立中興大學 (NCHU) × 國科會 TAIDE 8B 滿血 LoRA 微調工作台
本 Notebook 專為 Google Colab 免費 T4 GPU 設計，提供：
1. **國科會 Llama-3.1-TAIDE-LX-8B-Chat** 4-bit QLoRA 領域微調
2. 吃入 3,183 筆中興專屬指令集 (`train_instruct_v3.jsonl`)
3. 自動執行 7 大極端情境推理驗證（機車鎖車、特約診所、揚州炒飯等）
4. 自動合併權重並量化為 **4-bit GGUF (Q4_K_M)** 供邊緣端 / GitHub Pages / Ollama 部署

💡 請確認執行階段類型為 GPU：`執行階段 -> 變更執行階段類型 -> T4 GPU`。

### 步驟 1：檢查 GPU 資源（需 T4 或以上，約需 8GB 顯存）

In [ ]:
!nvidia-smi

### 步驟 2：一鍵安裝微調與量化環境依賴

In [ ]:
import os
try:
    # 優先從 Google Colab 左側 🔑 Secrets (密鑰) 讀取 HF_TOKEN
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("🔒 已成功從 Colab 密鑰載入 HF_TOKEN (無明碼洩漏風險)！")
    else:
        raise ValueError
except Exception:
    # 備援：若未設定 Colab Secrets，則透過安全環境變數或互動提示輸入
    from huggingface_hub import login
    if not os.environ.get("HF_TOKEN"):
        print("💡 建議至 Colab 左側 🔑 密鑰分頁新增 HF_TOKEN；或於下方安全輸入：")
        login()


### 步驟 3：登入 Hugging Face（TAIDE 8B 為 Gated 授權模型，需填入你的免費 Read Token）
若尚未取得 Token，可在 https://huggingface.co/settings/tokens 免費建立一個。

In [ ]:
import os
try:
    # 優先從 Google Colab 左側 🔑 Secrets (密鑰) 讀取 HF_TOKEN
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("🔒 已成功從 Colab 密鑰載入 HF_TOKEN (無明碼洩漏風險)！")
    else:
        raise ValueError
except Exception:
    # 備援：若未設定 Colab Secrets，則透過安全環境變數或互動提示輸入
    from huggingface_hub import login
    if not os.environ.get("HF_TOKEN"):
        print("💡 建議至 Colab 左側 🔑 密鑰分頁新增 HF_TOKEN；或於下方安全輸入：")
        login()


### 步驟 4：上傳並解壓縮訓練包 (`nchu_taide_training_bundle.zip`)
（檔案大小僅約 580 KB，可直接點擊執行上傳）

In [ ]:
import os
from google.colab import files

if not os.path.exists("nchu_taide_training_bundle.zip"):
    print("請上傳本機專案目錄下的 nchu_taide_training_bundle.zip：")
    uploaded = files.upload()

!unzip -o nchu_taide_training_bundle.zip
!ls -la train_taide_lora_colab.py *.jsonl

## 🚀 步驟 5：啟動 4-bit QLoRA 領域模型微調
- 模型：`taide/Llama-3.1-TAIDE-LX-8B-Chat`
- 預計時間：約 45 ~ 75 分鐘（T4 GPU，顯存佔用約 7.8GB）

In [ ]:
!python3 train_taide_lora_colab.py \
  --model_id taide/Llama-3.1-TAIDE-LX-8B-Chat \
  --output_name nchu_taide_adapter \
  --epochs 3 \
  --batch_size 2

## ⚡ 步驟 6：合併 LoRA 並執行 4-bit (Q4_K_M) GGUF 邊緣量化
自動編譯 llama.cpp 並產出 `nchu_taide_adapter_q4_k_m.gguf`（約 4.5GB，可直接丟進 Ollama 或 HF Space 跑）

In [ ]:
!bash export_nchu_taide_adapter_gguf.sh

### 步驟 7：下載訓練產物 (LoRA Adapter 包與 GGUF 量化模型)

In [ ]:
from google.colab import files

if os.path.exists("nchu_taide_adapter.zip"):
    print("📦 正在下載 nchu_taide_adapter.zip...")
    files.download("nchu_taide_adapter.zip")

if os.path.exists("nchu_taide_adapter_q4_k_m.gguf"):
    print("📦 正在下載 nchu_taide_adapter_q4_k_m.gguf...")
    files.download("nchu_taide_adapter_q4_k_m.gguf")